In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import sys
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)
import joblib

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import snntorch as snn
from snntorch import surrogate

sys.path.append(os.path.abspath(".."))
import config

FEATURES_DIR = os.path.join("..", config.FEATURES_DATA_DIR)
MODELS_DIR   = os.path.join("..", config.MODELS_DIR)
CKPT_DIR     = os.path.join("..", config.CHECKPOINTS_DIR)
METRICS_DIR  = os.path.join("..", config.OUTPUTS_METRICS)
PLOTS_DIR    = os.path.join("..", config.OUTPUTS_PLOTS)
FORECAST_DIR = os.path.join("..", config.OUTPUTS_FORECASTS)

for d in [MODELS_DIR, CKPT_DIR, METRICS_DIR,
          PLOTS_DIR, FORECAST_DIR]:
    os.makedirs(d, exist_ok=True)

CITIES      = list(config.CITIES.keys())
SEED        = config.SEED
PAST_HOURS  = 24
HORIZONS    = [1]
BATCH_SIZE  = 256
EPOCHS      = 50
LR          = config.LEARNING_RATE
TRAIN_FRAC  = config.TRAIN_FRAC
VAL_FRAC    = config.VAL_FRAC
DAYTIME_THR = config.DAYTIME_THR
TARGET      = "GHI"

# Smaller SNN hidden sizes for CPU speed
HIDDEN_SIZES = [256, 128, 64]

torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("✅ Imports ready")
print(f"   PyTorch      : {torch.__version__}")
print(f"   SNNTorch     : {snn.__version__}")
print(f"   Device       : {DEVICE}")
print(f"   PAST_HOURS   : {PAST_HOURS}")
print(f"   HORIZONS     : {HORIZONS}")
print(f"   HIDDEN_SIZES : {HIDDEN_SIZES}")

✅ Imports ready
   PyTorch      : 2.10.0+cpu
   SNNTorch     : 0.9.4
   Device       : cpu
   PAST_HOURS   : 24
   HORIZONS     : [1]
   HIDDEN_SIZES : [256, 128, 64]


In [2]:
# Exactly 15 features — same as Notebook 07
# This ensures fair comparison between BiLSTM and SNN
SEQUENCE_FEATURES = [
    "clear_sky_ghi",
    "solar_elevation",
    "cos_zenith",
    "hour_sin",
    "hour_cos",
    "doy_sin",
    "doy_cos",
    "temperature",
    "humidity",
    "wind_speed",
    "GHI_lag1",
    "GHI_lag24",
    "GHI_roll24_mean",
    "clearness_index",
    "is_daytime",
]

city_dfs = {}
print("📂 Loading feature CSVs...\n")
for city in CITIES:
    path = os.path.join(
        FEATURES_DIR, f"{city}_features.csv"
    )
    df = pd.read_csv(
        path, index_col="datetime", parse_dates=True
    )
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")
    city_dfs[city] = df
    print(f"  ✅ {city:<14}  shape={df.shape}")

sample_df         = city_dfs["new_delhi"]
SEQUENCE_FEATURES = [
    f for f in SEQUENCE_FEATURES
    if f in sample_df.columns
]
N_FEATURES = len(SEQUENCE_FEATURES)

print(f"\n✅ Sequence features : {N_FEATURES}")
print(f"   Must be 14 or 15 to match Notebook 07")
print(f"   Features: {SEQUENCE_FEATURES}")

📂 Loading feature CSVs...

  ✅ riyadh          shape=(26232, 58)
  ✅ cairo           shape=(26232, 58)
  ✅ istanbul        shape=(26232, 58)
  ✅ new_delhi       shape=(26232, 58)
  ✅ dubai           shape=(26232, 58)
  ✅ london          shape=(26232, 58)
  ✅ sydney          shape=(26232, 58)
  ✅ tokyo           shape=(26232, 58)
  ✅ los_angeles     shape=(26232, 58)
  ✅ nairobi         shape=(26232, 58)

✅ Sequence features : 15
   Must be 14 or 15 to match Notebook 07
   Features: ['clear_sky_ghi', 'solar_elevation', 'cos_zenith', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'temperature', 'humidity', 'wind_speed', 'GHI_lag1', 'GHI_lag24', 'GHI_roll24_mean', 'clearness_index', 'is_daytime']


In [4]:
def build_sequences_torch(
    df        : pd.DataFrame,
    features  : list,
    target    : str,
    past_hours: int,
    horizon   : int,
) -> tuple:
    feat_arr = df[features].values.astype(np.float32)
    tgt_arr  = df[target].values.astype(np.float32)
    n        = len(feat_arr)
    X, y     = [], []
    for i in range(past_hours, n - horizon + 1):
        X.append(feat_arr[i - past_hours : i])
        if horizon == 1:
            y.append(tgt_arr[i])
        else:
            y.append(tgt_arr[i : i + horizon])
    return (
        np.array(X, dtype=np.float32),
        np.array(y, dtype=np.float32),
    )


def prepare_city_torch(
    df        : pd.DataFrame,
    features  : list,
    target    : str,
    past_hours: int,
    horizon   : int,
):
    all_cols  = (
        [target] + [f for f in features if f != target]
    )
    scaler    = MinMaxScaler()
    scaled    = scaler.fit_transform(
        df[all_cols].values
    )
    df_sc     = pd.DataFrame(
        scaled, columns=all_cols, index=df.index
    )

    n         = len(df_sc)
    train_end = int(n * TRAIN_FRAC)
    val_end   = int(n * (TRAIN_FRAC + VAL_FRAC))
    feat_cols = [f for f in features if f != target]

    def make_tensors(subset):
        X, y = build_sequences_torch(
            subset, feat_cols, target,
            past_hours, horizon
        )
        if horizon == 1:
            yt = torch.tensor(
                y, dtype=torch.float32
            ).unsqueeze(-1)
        else:
            yt = torch.tensor(y, dtype=torch.float32)
        return torch.tensor(X, dtype=torch.float32), yt

    X_tr, y_tr = make_tensors(df_sc.iloc[:train_end])
    X_vl, y_vl = make_tensors(
        df_sc.iloc[train_end:val_end]
    )
    X_te, y_te = make_tensors(df_sc.iloc[val_end:])

    return X_tr, y_tr, X_vl, y_vl, X_te, y_te, scaler


def inverse_ghi(
    scaled_vals: np.ndarray,
    scaler     : MinMaxScaler,
) -> np.ndarray:
    n_cols      = scaler.n_features_in_
    dummy       = np.zeros((len(scaled_vals), n_cols))
    dummy[:, 0] = scaled_vals.flatten()
    return scaler.inverse_transform(dummy)[:, 0]


print("✅ Data pipeline defined")
print(f"   Train frac : {TRAIN_FRAC}")
print(f"   Val frac   : {VAL_FRAC}")
print(f"   Test frac  : {1 - TRAIN_FRAC - VAL_FRAC:.2f}")

✅ Data pipeline defined
   Train frac : 0.72
   Val frac   : 0.08
   Test frac  : 0.20


In [5]:
class NeuroSpikeSNN(nn.Module):
    """
    Spiking Neural Network for Solar Irradiance Forecasting.

    Key SNN concepts:
    - LIF neuron: membrane potential accumulates input
      until threshold is reached, then fires a spike
      and resets. This mimics biological neurons.
    - Surrogate gradient: since the spike function
      (Heaviside step) is non-differentiable, we use
      fast sigmoid as a smooth approximation during
      backpropagation.
    - Beta (membrane decay): controls how fast the
      membrane potential leaks between time steps.
      Learned per layer during training.
    - Spike rate readout: average firing rate over
      simulation steps encodes the prediction value.
    """

    def __init__(
        self,
        n_time_steps : int,
        n_features   : int,
        horizon      : int   = 1,
        hidden_sizes : list  = None,
        beta         : float = 0.95,
        dropout_rate : float = 0.2,
    ):
        super(NeuroSpikeSNN, self).__init__()

        if hidden_sizes is None:
            hidden_sizes = [256, 128, 64]

        self.n_time_steps = n_time_steps
        self.n_features   = n_features
        self.horizon      = horizon
        input_dim         = n_time_steps * n_features
        spike_grad        = surrogate.fast_sigmoid(
            slope=25
        )

        # ── Encoder ───────────────────────────────────────────
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_sizes[0]),
            nn.LayerNorm(hidden_sizes[0]),
        )

        # ── LIF Layer 1 ───────────────────────────────────────
        self.fc1   = nn.Linear(
            hidden_sizes[0], hidden_sizes[0]
        )
        self.lif1  = snn.Leaky(
            beta=beta, spike_grad=spike_grad,
            learn_beta=True, threshold=1.0,
        )
        self.drop1 = nn.Dropout(dropout_rate)

        # ── LIF Layer 2 ───────────────────────────────────────
        self.fc2   = nn.Linear(
            hidden_sizes[0], hidden_sizes[1]
        )
        self.lif2  = snn.Leaky(
            beta=beta, spike_grad=spike_grad,
            learn_beta=True, threshold=1.0,
        )
        self.drop2 = nn.Dropout(dropout_rate)

        # ── LIF Layer 3 ───────────────────────────────────────
        self.fc3   = nn.Linear(
            hidden_sizes[1], hidden_sizes[2]
        )
        self.lif3  = snn.Leaky(
            beta=beta, spike_grad=spike_grad,
            learn_beta=True, threshold=1.0,
        )

        # ── Decoder ───────────────────────────────────────────
        self.decoder = nn.Sequential(
            nn.Linear(hidden_sizes[2], 32),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, horizon),
        )

    def forward(self, x: torch.Tensor) -> tuple:
        batch     = x.shape[0]
        x_flat    = x.reshape(batch, -1)
        encoded   = self.encoder(x_flat)

        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()

        s1_rec, s2_rec, s3_rec = [], [], []
        cur = encoded

        for _ in range(10):
            spk1, mem1 = self.lif1(self.fc1(cur),   mem1)
            spk1       = self.drop1(spk1)
            s1_rec.append(spk1)

            spk2, mem2 = self.lif2(self.fc2(spk1),  mem2)
            spk2       = self.drop2(spk2)
            s2_rec.append(spk2)

            spk3, mem3 = self.lif3(self.fc3(spk2),  mem3)
            s3_rec.append(spk3)

        rate   = torch.stack(s3_rec, dim=0).mean(dim=0)
        output = self.decoder(rate)

        spikes = {
            "layer1": torch.stack(s1_rec, dim=0),
            "layer2": torch.stack(s2_rec, dim=0),
            "layer3": torch.stack(s3_rec, dim=0),
        }
        return output, spikes


# ── Verify architecture ───────────────────────────────────────
test_snn = NeuroSpikeSNN(
    n_time_steps = PAST_HOURS,
    n_features   = N_FEATURES - 1,
    horizon      = 1,
    hidden_sizes = HIDDEN_SIZES,
).to(DEVICE)

n_params = sum(
    p.numel() for p in test_snn.parameters()
    if p.requires_grad
)

print("✅ NeuroSpike SNN defined")
print(f"   Hidden sizes         : {HIDDEN_SIZES}")
print(f"   Trainable parameters : {n_params:,}")
print(f"   Device               : {DEVICE}")
print(test_snn)

✅ NeuroSpike SNN defined
   Hidden sizes         : [256, 128, 64]
   Trainable parameters : 196,356
   Device               : cpu
NeuroSpikeSNN(
  (encoder): Sequential(
    (0): Linear(in_features=336, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (fc1): Linear(in_features=256, out_features=256, bias=True)
  (lif1): Leaky()
  (drop1): Dropout(p=0.2, inplace=False)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (lif2): Leaky()
  (drop2): Dropout(p=0.2, inplace=False)
  (fc3): Linear(in_features=128, out_features=64, bias=True)
  (lif3): Leaky()
  (decoder): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): ReLU()
    (5): Linear(in_features=16, out_features=1, bias=True)
  )
)


In [6]:
def compute_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    label : str = "",
) -> dict:
    y_true    = np.clip(y_true, 0, None)
    y_pred    = np.clip(y_pred, 0, None)
    rmse      = np.sqrt(mean_squared_error(y_true, y_pred))
    mae       = mean_absolute_error(y_true, y_pred)
    r2        = r2_score(y_true, y_pred)
    mask      = y_true > 1.0
    mape      = (
        np.mean(np.abs(
            (y_true[mask] - y_pred[mask]) / y_true[mask]
        )) * 100
        if mask.sum() > 0 else np.nan
    )
    pers      = np.roll(y_true, 1)
    pers[0]   = y_true[0]
    rmse_p    = np.sqrt(mean_squared_error(y_true, pers))
    skill     = (1 - rmse / rmse_p) * 100 if rmse_p > 0 else 0
    return {
        "model" : label,
        "RMSE"  : round(float(rmse),  3),
        "MAE"   : round(float(mae),   3),
        "MAPE"  : round(float(mape),  3),
        "R2"    : round(float(r2),    4),
        "Skill" : round(float(skill), 2),
    }


def train_one_epoch(
    model, loader, optimizer, criterion
) -> float:
    model.train()
    total = 0.0
    for X_b, y_b in loader:
        X_b = X_b.to(DEVICE)
        y_b = y_b.to(DEVICE)
        optimizer.zero_grad()
        out, _ = model(X_b)
        loss   = criterion(out, y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(
            model.parameters(), 1.0
        )
        optimizer.step()
        total += loss.item()
    return total / len(loader)


def evaluate(
    model, loader, criterion
) -> tuple:
    model.eval()
    total     = 0.0
    all_preds = []
    all_trues = []
    with torch.no_grad():
        for X_b, y_b in loader:
            X_b    = X_b.to(DEVICE)
            y_b    = y_b.to(DEVICE)
            out, _ = model(X_b)
            loss   = criterion(out, y_b)
            total += loss.item()
            all_preds.append(out.cpu().numpy())
            all_trues.append(y_b.cpu().numpy())
    preds = np.concatenate(all_preds, axis=0)
    trues = np.concatenate(all_trues, axis=0)
    return total / len(loader), preds, trues


print("✅ Training utilities defined")

✅ Training utilities defined


In [12]:
all_snn_results = []
snn_histories   = {}

print("=" * 65)
print("  TRAINING NeuroSpike SNN — ALL CITIES")
print("=" * 65)

for city in CITIES:
    print(f"\n{'─'*55}")
    print(f"  📍 {city.upper()}")
    print(f"{'─'*55}")

    df = city_dfs[city]

    for horizon in HORIZONS:
        print(f"\n  ⚡ Horizon = {horizon}h ahead")

        (X_tr, y_tr, X_vl, y_vl,
         X_te, y_te, scaler) = prepare_city_torch(
            df, SEQUENCE_FEATURES, TARGET,
            PAST_HOURS, horizon,
        )

        print(
            f"     Train:{X_tr.shape}  "
            f"Val:{X_vl.shape}  "
            f"Test:{X_te.shape}"
        )

        train_loader = DataLoader(
            TensorDataset(X_tr, y_tr),
            batch_size=BATCH_SIZE, shuffle=True,
        )
        val_loader = DataLoader(
            TensorDataset(X_vl, y_vl),
            batch_size=BATCH_SIZE, shuffle=False,
        )
        test_loader = DataLoader(
            TensorDataset(X_te, y_te),
            batch_size=BATCH_SIZE, shuffle=False,
        )

        model = NeuroSpikeSNN(
            n_time_steps = PAST_HOURS,
            n_features   = X_tr.shape[2],
            horizon      = horizon,
            hidden_sizes = HIDDEN_SIZES,
        ).to(DEVICE)

        optimizer = optim.AdamW(
            model.parameters(),
            lr=LR, weight_decay=1e-4,
        )
        criterion = nn.HuberLoss()
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, factor=0.5, patience=7,
            min_lr=1e-7,
        )

        best_val  = float("inf")
        patience  = 0
        PAT_MAX   = 15
        tr_losses = []
        vl_losses = []
        ckpt_path = os.path.join(
            CKPT_DIR, f"snn_{city}_h{horizon}.pt"
        )

        for epoch in range(1, EPOCHS + 1):
            tr       = train_one_epoch(
                model, train_loader, optimizer, criterion
            )
            vl, _, _ = evaluate(
                model, val_loader, criterion
            )
            scheduler.step(vl)
            tr_losses.append(tr)
            vl_losses.append(vl)

            if vl < best_val:
                best_val  = vl
                patience  = 0
                torch.save(model.state_dict(), ckpt_path)
            else:
                patience += 1

            if epoch % 10 == 0:
                print(
                    f"     Epoch {epoch:>3}  "
                    f"train={tr:.5f}  "
                    f"val={vl:.5f}  "
                    f"pat={patience}/{PAT_MAX}"
                )

            if patience >= PAT_MAX:
                print(f"     ⏹ Early stop epoch {epoch}")
                break

        snn_histories[f"{city}_h{horizon}"] = {
            "loss"    : tr_losses,
            "val_loss": vl_losses,
        }

        model.load_state_dict(
            torch.load(ckpt_path, map_location=DEVICE)
        )
        _, preds_sc, trues_sc = evaluate(
            model, test_loader, criterion
        )

        if horizon == 1:
            y_pred = np.clip(
                inverse_ghi(preds_sc.flatten(), scaler),
                0, None,
            )
            y_true = inverse_ghi(
                trues_sc.flatten(), scaler
            )
            m = compute_metrics(
                y_true, y_pred,
                f"NeuroSpike_h{horizon}",
            )
            m["city"]    = city
            m["horizon"] = horizon
        else:
            steps = []
            for step in range(horizon):
                yt = inverse_ghi(
                    trues_sc[:, step], scaler
                )
                yp = np.clip(
                    inverse_ghi(
                        preds_sc[:, step], scaler
                    ),
                    0, None,
                )
                steps.append(compute_metrics(yt, yp))
            m = {
                "model"  : f"NeuroSpike_h{horizon}",
                "city"   : city,
                "horizon": horizon,
                "RMSE"   : round(np.mean(
                    [s["RMSE"]  for s in steps]), 3),
                "MAE"    : round(np.mean(
                    [s["MAE"]   for s in steps]), 3),
                "R2"     : round(np.mean(
                    [s["R2"]    for s in steps]), 4),
                "Skill"  : round(np.mean(
                    [s["Skill"] for s in steps]), 2),
                "MAPE"   : np.nan,
            }

        all_snn_results.append(m)

        torch.save(
            model.state_dict(),
            os.path.join(
                MODELS_DIR,
                f"neurospike_{city}_h{horizon}.pt",
            ),
        )

        print(
            f"     ✅ RMSE={m['RMSE']:.3f}  "
            f"MAE={m['MAE']:.3f}  "
            f"R²={m['R2']:.4f}  "
            f"Skill={m['Skill']:.1f}%"
        )

print(f"\n{'='*65}")
print(
    f"  ✅ SNN training complete — "
    f"{len(all_snn_results)} results"
)

  TRAINING NeuroSpike SNN — ALL CITIES

───────────────────────────────────────────────────────
  📍 RIYADH
───────────────────────────────────────────────────────

  ⚡ Horizon = 1h ahead
     Train:torch.Size([18863, 24, 15])  Val:torch.Size([2074, 24, 15])  Test:torch.Size([5223, 24, 15])
     Epoch  10  train=0.00047  val=0.00089  pat=0/15
     Epoch  20  train=0.00032  val=0.00083  pat=1/15
     Epoch  30  train=0.00024  val=0.00087  pat=11/15
     ⏹ Early stop epoch 34
     ✅ RMSE=23.905  MAE=16.282  R²=0.9950  Skill=78.4%

───────────────────────────────────────────────────────
  📍 CAIRO
───────────────────────────────────────────────────────

  ⚡ Horizon = 1h ahead
     Train:torch.Size([18863, 24, 15])  Val:torch.Size([2074, 24, 15])  Test:torch.Size([5223, 24, 15])
     Epoch  10  train=0.00077  val=0.00155  pat=0/15
     Epoch  20  train=0.00049  val=0.00122  pat=2/15
     Epoch  30  train=0.00034  val=0.00081  pat=0/15
     Epoch  40  train=0.00031  val=0.00070  pat=0/15
    

In [13]:
snn_df   = pd.DataFrame(all_snn_results)
snn_path = os.path.join(METRICS_DIR, "snn_results.csv")
snn_df.to_csv(snn_path, index=False)

base_df   = pd.read_csv(
    os.path.join(METRICS_DIR, "baseline_results.csv")
)
lstm_path = os.path.join(METRICS_DIR, "lstm_results.csv")
lstm_df   = (
    pd.read_csv(lstm_path)
    if os.path.exists(lstm_path)
    else pd.DataFrame()
)

print("=" * 70)
print("  FULL MODEL COMPARISON — h=1 — ALL CITIES")
print("=" * 70)
print(
    f"\n  {'City':<14} {'Persistence':>12} "
    f"{'XGBoost':>10} {'BiLSTM':>10} "
    f"{'NeuroSpike':>12}"
)
print(
    f"  {'─'*14} {'─'*12} {'─'*10} "
    f"{'─'*10} {'─'*12}"
)

for city in CITIES:
    def get_r(df, pat, h=None):
        if len(df) == 0:
            return np.nan
        mask = df["model"].str.contains(pat, na=False)
        if h is not None and "horizon" in df.columns:
            mask = mask & (df["horizon"] == h)
        rows = df[mask & (df["city"] == city)]
        return (rows["RMSE"].values[0]
                if len(rows) > 0 else np.nan)

    p = get_r(base_df, "Persistence")
    x = get_r(base_df, "XGBoost")
    l = get_r(lstm_df, "BiLSTM", 1)
    s = get_r(snn_df,  "NeuroSpike", 1)

    vals = [v for v in [p, x, l, s]
            if not np.isnan(v)]
    best = min(vals) if vals else np.nan
    flag = lambda v: "🏆" if (
        not np.isnan(v) and v == best
    ) else "  "

    print(
        f"  {city:<14} "
        f"{p:>10.2f}   "
        f"{x:>8.2f}   "
        f"{l:>8.2f}   "
        f"{s:>10.2f} {flag(s)}"
    )

print(f"\n💾 SNN results → {snn_path}")

  FULL MODEL COMPARISON — h=1 — ALL CITIES

  City            Persistence    XGBoost     BiLSTM   NeuroSpike
  ────────────── ──────────── ────────── ────────── ────────────
  riyadh             110.82      25.74      38.09        23.91 🏆
  cairo              102.99      31.77      40.57        28.02 🏆
  istanbul            79.87      48.91      37.68        27.74 🏆
  new_delhi           86.67      58.29      54.44        35.31 🏆
  dubai               99.19      27.16      34.70        26.99 🏆
  london              58.30      52.88      41.41        28.27 🏆
  sydney              92.63      68.09      60.49        34.04 🏆
  tokyo               80.95      60.01      46.02        30.60 🏆
  los_angeles         97.53      39.43      49.66        27.44 🏆
  nairobi            104.78      60.53      52.12        34.35 🏆

💾 SNN results → ..\outputs/metrics\snn_results.csv


In [14]:
city    = "new_delhi"
horizon = 1
df      = city_dfs[city]

(_, _, _, _, X_te, y_te,
 scaler) = prepare_city_torch(
    df, SEQUENCE_FEATURES, TARGET,
    PAST_HOURS, horizon,
)

model = NeuroSpikeSNN(
    n_time_steps = PAST_HOURS,
    n_features   = X_te.shape[2],
    horizon      = horizon,
    hidden_sizes = HIDDEN_SIZES,
).to(DEVICE)

model.load_state_dict(torch.load(
    os.path.join(CKPT_DIR,
                 f"snn_{city}_h{horizon}.pt"),
    map_location=DEVICE,
))
model.eval()

sample_X = X_te[:64].to(DEVICE)
with torch.no_grad():
    output, spikes = model(sample_X)

spk1  = spikes["layer1"].cpu().numpy()
spk2  = spikes["layer2"].cpu().numpy()
spk3  = spikes["layer3"].cpu().numpy()
rate1 = float(spk1.mean())
rate2 = float(spk2.mean())
rate3 = float(spk3.mean())

fig = plt.figure(figsize=(16, 12))
fig.suptitle(
    f"NeuroSpike — Spike Activity\n"
    f"{city.replace('_',' ').title()}",
    fontsize=13, fontweight="bold",
)
gs = gridspec.GridSpec(
    3, 2, figure=fig,
    hspace=0.45, wspace=0.35,
)

# ── Raster plot ───────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
n_neurons = min(20, spk1.shape[2])
for nid in range(n_neurons):
    col   = spk1[:, :, nid].reshape(-1)
    t_idx = np.where(col > 0.5)[0]
    ax1.scatter(
        t_idx,
        np.full_like(t_idx, nid),
        s=1, color="#4A90E8", alpha=0.6,
    )
ax1.set_xlabel("Simulation step × sample")
ax1.set_ylabel("Neuron index")
ax1.set_title(
    f"Layer 1 spike raster "
    f"({n_neurons} neurons, 64 samples) "
    f"— rate={rate1:.3f}"
)
ax1.grid(True, alpha=0.2)

# ── Firing rates per layer ────────────────────────────────────
ax2    = fig.add_subplot(gs[1, 0])
labels = ["L1 (256)", "L2 (128)", "L3 (64)"]
rates  = [rate1, rate2, rate3]
colors = ["#4A90E8", "#E8883A", "#4ABA6A"]
bars   = ax2.bar(labels, rates, color=colors, alpha=0.85)
ax2.set_ylabel("Mean spike rate")
ax2.set_title("Firing rate per layer")
ax2.set_ylim(0, 1)
ax2.grid(True, alpha=0.3, axis="y")
for bar, val in zip(bars, rates):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{val:.3f}", ha="center", fontsize=10,
    )

# ── Spike distribution L3 ─────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
ax3.hist(
    spk3.flatten(), bins=30,
    color="#4ABA6A", alpha=0.7,
    edgecolor="white", linewidth=0.3,
)
ax3.set_xlabel("Spike value")
ax3.set_ylabel("Count")
ax3.set_title("Layer 3 spike distribution")
ax3.grid(True, alpha=0.3)

# ── Prediction scatter ────────────────────────────────────────
ax4      = fig.add_subplot(gs[2, 0])
pred_np  = output.cpu().numpy().flatten()
true_np  = y_te[:64].numpy().flatten()
y_pred_p = np.clip(inverse_ghi(pred_np, scaler), 0, None)
y_true_p = inverse_ghi(true_np, scaler)
ax4.scatter(
    y_true_p, y_pred_p,
    s=8, alpha=0.5, color="#9B59B6",
)
lim = max(y_true_p.max(), y_pred_p.max())
ax4.plot([0, lim], [0, lim], "k--", lw=1, alpha=0.6)
ax4.set_xlabel("Actual GHI (W/m²)")
ax4.set_ylabel("Predicted GHI (W/m²)")
ax4.set_title("SNN predictions — sample batch")
ax4.grid(True, alpha=0.3)

# ── Training history ──────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 1])
key = f"{city}_h{horizon}"
if key in snn_histories:
    h = snn_histories[key]
    ax5.plot(h["loss"],
             color="#4A90E8", lw=1.5, label="Train")
    ax5.plot(h["val_loss"],
             color="#E84A4A", lw=1.5, label="Val")
    ax5.set_xlabel("Epoch")
    ax5.set_ylabel("Huber loss")
    ax5.set_title("Training history")
    ax5.legend()
    ax5.grid(True, alpha=0.3)

plt.tight_layout()
path = os.path.join(PLOTS_DIR, "08_snn_spike_analysis.png")
plt.savefig(path, dpi=120, bbox_inches="tight")
plt.show()
print(f"💾 Saved → {path}")
print(f"\n   L1 spike rate : {rate1:.4f}")
print(f"   L2 spike rate : {rate2:.4f}")
print(f"   L3 spike rate : {rate3:.4f}")

💾 Saved → ..\outputs/plots\08_snn_spike_analysis.png

   L1 spike rate : 0.2661
   L2 spike rate : 0.1969
   L3 spike rate : 0.1610


In [15]:
city    = "new_delhi"
horizon = 1
df      = city_dfs[city]

(_, _, _, _, X_te, y_te,
 scaler) = prepare_city_torch(
    df, SEQUENCE_FEATURES, TARGET,
    PAST_HOURS, horizon,
)

model = NeuroSpikeSNN(
    n_time_steps = PAST_HOURS,
    n_features   = X_te.shape[2],
    horizon      = horizon,
    hidden_sizes = HIDDEN_SIZES,
).to(DEVICE)

model.load_state_dict(torch.load(
    os.path.join(CKPT_DIR,
                 f"snn_{city}_h{horizon}.pt"),
    map_location=DEVICE,
))
model.eval()

test_loader = DataLoader(
    TensorDataset(X_te, y_te),
    batch_size=BATCH_SIZE, shuffle=False,
)
_, preds_sc, trues_sc = evaluate(
    model, test_loader, nn.HuberLoss()
)

y_pred = np.clip(
    inverse_ghi(preds_sc.flatten(), scaler), 0, None
)
y_true = inverse_ghi(trues_sc.flatten(), scaler)
n_plot = 24 * 5

fig, axes = plt.subplots(
    2, 1, figsize=(16, 8), sharex=True
)
fig.suptitle(
    f"NeuroSpike SNN — Predictions vs Actual\n"
    f"{city.replace('_',' ').title()} — 5 Days",
    fontsize=12, fontweight="bold",
)
t = np.arange(n_plot)

axes[0].plot(
    t, y_true[:n_plot],
    color="black", lw=1.5, label="Actual GHI",
)
axes[0].plot(
    t, y_pred[:n_plot],
    color="#9B59B6", lw=1.2, linestyle="--",
    label="NeuroSpike predicted", alpha=0.85,
)
axes[0].set_ylabel("GHI (W/m²)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].fill_between(
    t,
    np.abs(y_pred[:n_plot] - y_true[:n_plot]),
    color="#E84A4A", alpha=0.5, label="|Error|",
)
axes[1].set_ylabel("Absolute error (W/m²)")
axes[1].set_xlabel("Hour")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
path = os.path.join(PLOTS_DIR, "08_snn_predictions.png")
plt.savefig(path, dpi=120, bbox_inches="tight")
plt.show()

m = compute_metrics(y_true, y_pred, "NeuroSpike")
print(f"\n📊 NeuroSpike — New Delhi h=1:")
print(f"   RMSE  = {m['RMSE']:.3f} W/m²")
print(f"   MAE   = {m['MAE']:.3f} W/m²")
print(f"   R²    = {m['R2']:.4f}")
print(f"   Skill = {m['Skill']:.1f}% vs persistence")
print(f"💾 Saved → {path}")


📊 NeuroSpike — New Delhi h=1:
   RMSE  = 35.313 W/m²
   MAE   = 17.829 W/m²
   R²    = 0.9806
   Skill = 59.2% vs persistence
💾 Saved → ..\outputs/plots\08_snn_predictions.png


In [18]:
print("=" * 65)
print("  NOTEBOOK 08 — COMPLETE")
print("=" * 65)

h1 = snn_df[snn_df["horizon"] == 1]
print(f"\n  NeuroSpike SNN h=1 — all cities:")
print(
    f"  {'City':<14} {'RMSE':>8} "
    f"{'MAE':>8} {'R²':>8} {'Skill':>8}"
)
print(
    f"  {'─'*14} {'─'*8} "
    f"{'─'*8} {'─'*8} {'─'*8}"
)

for _, row in h1.iterrows():
    print(
        f"  {row['city']:<14} "
        f"{row['RMSE']:>8.3f} "
        f"{row['MAE']:>8.3f} "
        f"{row['R2']:>8.4f} "
        f"{row['Skill']:>7.1f}%"
    )

print(f"\n  Mean RMSE  : {h1['RMSE'].mean():.3f} W/m²")
print(f"  Mean Skill : {h1['Skill'].mean():.1f}%")

saved = [
    f for f in os.listdir(MODELS_DIR)
    if f.startswith("neurospike")
]
print(f"\n  Models saved : {len(saved)}")
for f in sorted(saved):
    print(f"    📁 {f}")

print(f"\n  Ready for → 09_evaluation.ipynb")

  NOTEBOOK 08 — COMPLETE

  NeuroSpike SNN h=1 — all cities:
  City               RMSE      MAE       R²    Skill
  ────────────── ──────── ──────── ──────── ────────
  riyadh           23.905   16.282   0.9950    78.4%
  cairo            28.019   15.860   0.9924    72.8%
  istanbul         27.742   16.922   0.9888    65.3%
  new_delhi        35.313   17.829   0.9806    59.2%
  dubai            26.992   15.153   0.9916    72.8%
  london           28.267   13.805   0.9797    51.3%
  sydney           34.041   17.691   0.9853    63.3%
  tokyo            30.602   17.422   0.9841    62.2%
  los_angeles      27.437   21.592   0.9921    71.8%
  nairobi          34.351   24.304   0.9873    67.2%

  Mean RMSE  : 29.667 W/m²
  Mean Skill : 66.4%

  Models saved : 10
    📁 neurospike_cairo_h1.pt
    📁 neurospike_dubai_h1.pt
    📁 neurospike_istanbul_h1.pt
    📁 neurospike_london_h1.pt
    📁 neurospike_los_angeles_h1.pt
    📁 neurospike_nairobi_h1.pt
    📁 neurospike_new_delhi_h1.pt
    📁 neurospi